# True Crime Pipeline — Colab GPU Runner

Edit code in **Cursor**, push to GitHub, then test in Colab.

1. **Runtime → Change runtime type → GPU** (A100 if your Colab plan provides it)
2. Run **Setup** cells once per Colab session (clone + pip; optional Drive cache; load models)
3. **Topic selection smoke test** — fast, no GPU models; run many times to verify page picking
4. **Sync & test run** — full pipeline after topic selection looks good

## Setup (once per Colab session)

In [ ]:
import os

REPO_URL = "https://github.com/Paarth-Rana/Agent-Based-TrueCrime-Content-Generation-Pipeline.git"
REPO_DIR = "/content/Agent-Based-TrueCrime-Content-Generation-Pipeline"
BRANCH = "main"

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL}
else:
    %cd {REPO_DIR}
    !git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}

%cd {REPO_DIR}
!pip install -q -r requirements.txt
print("Repo ready:", REPO_DIR)

In [ ]:
# Optional: persist Hugging Face downloads across Colab disconnects
USE_DRIVE_CACHE = True  # set False to skip

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
    os.makedirs(os.environ["HF_HOME"], exist_ok=True)
    print("HF_HOME:", os.environ["HF_HOME"])
else:
    print("Using default Colab HF cache (cleared when runtime disconnects)")

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Warning: no GPU — pipeline will be very slow on CPU.")

In [ ]:
# Loads Qwen + Bark + SDXL (several minutes first time; uses HF cache if set)
%cd /content/Agent-Based-TrueCrime-Content-Generation-Pipeline

import pipeline
print("Models loaded. Ready for test runs.")

## Topic selection smoke test (fast)

Runs **only** Wikipedia topic picking — no GPU models, no script/images/audio. Use this to verify curated case selection before a full pipeline run.

After changing `topic.py`, run `git pull` then reload `topic` only. Takes seconds.

In [ ]:
import importlib

N = 20          # number of trials
TOPIC = ""      # "" = random curated; or e.g. "Ted Kaczynski"
BRANCH = "main"

%cd /content/Agent-Based-TrueCrime-Content-Generation-Pipeline
!git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}

import topic
importlib.reload(topic)

rows = []
for i in range(N):
    state = topic.discover_topic(TOPIC)
    check = topic.validate_selection(state)
    rows.append({
        "run": i + 1,
        "title": state.get("source_title", ""),
        "search_query": state.get("search_query", ""),
        "url": state.get("source_url", ""),
        "ok": check["ok"],
    })

passed = sum(1 for r in rows if r["ok"])
unique_titles = sorted({r["title"] for r in rows if r["title"]})

print(f"Pass rate: {passed}/{N}")
print(f"Unique titles: {len(unique_titles)}")
print()
print(f"{'Run':<4} {'OK':<4} {'Title'}")
print("-" * 60)
for r in rows:
    mark = "yes" if r["ok"] else "NO"
    print(f"{r['run']:<4} {mark:<4} {r['title']}")

if passed < N:
    print("\nFailures:")
    for r in rows:
        if not r["ok"]:
            print(f"  run {r['run']}: {r['title']} — {r['url']}")

## Sync & test run (after each Cursor edit + `git push`)

Do **not** restart runtime for normal code changes. Only re-run setup if you changed `requirements.txt` or hit OOM.

In [ ]:
import importlib
import utils
import topic
import pipeline

TOPIC = ""  # or e.g. "D. B. Cooper" for a guided topic
BRANCH = "main"

%cd /content/Agent-Based-TrueCrime-Content-Generation-Pipeline
!git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}

# Reload utils + topic before pipeline (pipeline imports both)
importlib.reload(utils)
importlib.reload(topic)
importlib.reload(pipeline)

state = pipeline.run_pipeline(TOPIC)
print("Chosen:", state.get("source_title"))
print("Wiki:", state.get("source_url"))
print("Output folder:", state.get("out_dir"))
print("Manifest:", state.get("manifest_path"))

## Download outputs (optional)

In [ ]:
from google.colab import files

!zip -r /content/truecrime_outputs.zip outputs/
files.download("/content/truecrime_outputs.zip")